##CNNs for Text Classifications ✍

**Objective. What’s fundamentally different about text vs images?**

- tokenization → vocabulary → padding/truncation → OOV handling
- kernel size ↔ n-gram length; **global max-over-time pooling**
- learned vs. (optional) pre-trained embeddings; freeze vs. finetune
- text-specific ablations (kernels, pooling, max_len, embedding dim)

**Write (2–3 bullets):** Two ways text differs from images for CNNs (representation, invariances, preprocessing).

In [ ]:
# Core
import math, time, random, numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW, SGD

# Plots (optional)
import matplotlib.pyplot as plt

# Repro + device
SEED = globals().get("SEED", 42)
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# --- Config (tweak as you like) ---
DS_NAME      = "ag_news"   # or "imdb"
VALID_SPLIT  = 0.10
MAX_LEN      = 128
BATCH_SIZE   = 128
VOCAB_SIZE   = 30000       # typical modern size
SEED         = 42


Device: cuda


In [ ]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split
import numpy as np, random, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed(SEED)

raw = load_dataset(DS_NAME)
if DS_NAME == "ag_news":
    train_texts = [ex["text"]  for ex in raw["train"]]
    train_labels= [ex["label"] for ex in raw["train"]]
    test_texts  = [ex["text"]  for ex in raw["test"]]
    test_labels = [ex["label"] for ex in raw["test"]]
    num_classes = 4
elif DS_NAME == "imdb":
    train_texts = [ex["text"]  for ex in raw["train"]]
    train_labels= [int(ex["label"]) for ex in raw["train"]]
    test_texts  = [ex["text"]  for ex in raw["test"]]
    test_labels = [int(ex["label"]) for ex in raw["test"]]
    num_classes = 2
else:
    raise ValueError("Supported: 'ag_news' or 'imdb'")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=VALID_SPLIT, random_state=SEED, stratify=train_labels
)
print(f"{DS_NAME} → train={len(train_texts)}  val={len(val_texts)}  test={len(test_texts)}  classes={num_classes}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

ag_news → train=108000  val=12000  test=7600  classes=4


## Data pipeline (tokenize - vocab - pad)

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import WordPieceTrainer

SPECIAL_TOKENS = ["[PAD]", "[UNK]"]

# 1) Create blank WordPiece tokenizer
wp_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
wp_tokenizer.pre_tokenizer = Whitespace()

# 2) Train on training texts
trainer = WordPieceTrainer(vocab_size=VOCAB_SIZE, special_tokens=SPECIAL_TOKENS)
wp_tokenizer.train_from_iterator(train_texts, trainer)

# 3) Vocab + indices
token2id = wp_tokenizer.get_vocab()            # str -> int
id2token = {i:t for t,i in token2id.items()}
PAD_IDX  = token2id["[PAD]"]
UNK_IDX  = token2id["[UNK]"]
vocab_size = len(token2id)

print("Tokenizer: WordPiece")
print("vocab_size:", vocab_size, "| PAD_IDX:", PAD_IDX, "| UNK_IDX:", UNK_IDX)

# (Optional) Save/reload for reproducibility:
# wp_tokenizer.save("wordpiece_tokenizer.json")
# wp_tokenizer = Tokenizer.from_file("wordpiece_tokenizer.json")


Tokenizer: WordPiece
vocab_size: 30000 | PAD_IDX: 0 | UNK_IDX: 1


In [ ]:
def encode_ids(text, max_len=MAX_LEN):
    ids = wp_tokenizer.encode(text).ids
    if len(ids) > max_len:
        ids = ids[:max_len]
    if len(ids) < max_len:
        ids = ids + [PAD_IDX] * (max_len - len(ids))
    return ids

def decode_ids(ids):
    # For visualization (labels in plots, etc.)
    # tokenizers library also supports .decode, but it's slower; for debugging tokens, use id2token:
    return [id2token.get(i, "[UNK]") for i in ids]


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TextDatasetWP(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = texts, labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        return int(self.labels[i]), torch.tensor(encode_ids(self.texts[i]), dtype=torch.long)

def collate_wp(batch):
    labels, ids = zip(*batch)
    return torch.tensor(labels, dtype=torch.long), torch.stack(ids)

train_ds = TextDatasetWP(train_texts, train_labels)
val_ds   = TextDatasetWP(val_texts,   val_labels)
test_ds  = TextDatasetWP(test_texts,  test_labels)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_wp)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_wp)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_wp)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Loaders ready • device:", device)


Loaders ready • device: cuda


In [ ]:
train_texts[0]

'10 seconds that change everything ATHENS - Ten seconds. Barely time enough to tie a shoe, wash a glass, get the paper off the porch. But when the moment comes, and eight men kneel at their blocks and peer down the empty, waiting track, it is as if the entire Olympics stop to watch.'

In [ ]:
x[0]

tensor([ 7556, 16305,    12,  1683,  4185, 21872,   178, 16676, 16797,    12,
        10045,  1622, 13327, 21872,    11,  6043,    11,   713, 18221, 28146,
           12, 11596,     4,   217,    26,    75,  1686,  9671,   178,   172,
         1650,  1683,  1255,    11,   263,   178,    57, 16676,   345,  6981,
           57,  5647,    11,   236,  2788, 26769,   279,   646,    13,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0])

In [ ]:
y, x = next(iter(train_loader))
print("labels:", y.shape, "tokens:", x.shape, " (B, L) expected →", (BATCH_SIZE, MAX_LEN))
print("first 15 ids:", x[0, :15].tolist())
print("first 15 toks:", decode_ids(x[0, :15].tolist()))


labels: torch.Size([128]) tokens: torch.Size([128, 128])  (B, L) expected → (128, 128)
first 15 ids: [7556, 16305, 12, 1683, 4185, 21872, 178, 16676, 16797, 12, 10045, 1622, 13327, 21872, 11]
first 15 toks: ['Philippine', 'actor', '-', 'presidential', 'candidate', 'Poe', 'in', 'coma', 'MANILA', '-', 'Movie', 'star', 'Fernando', 'Poe', ',']


## Baseline TextCNN (k=3, global max pool)

In [ ]:
class TextCNN(nn.Module):
    """Embedding -> Conv1d(k=3) -> ReLU -> Global Max Pool -> Linear"""
    def __init__(self, vocab_size, embed_dim=100, num_filters=100, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        # list layers one after another (no conditionals)
        self.conv = nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=3, padding=1)
        self.fc   = nn.Linear(num_filters, num_classes)

    def forward(self, batch_tokens):
        x = self.embedding(batch_tokens)      # (B, T, E)
        x = x.transpose(1, 2)                 # (B, E, T) for Conv1d over time
        x = F.relu(self.conv(x))              # (B, C, T)
        x = torch.max(x, dim=2).values        # global max-over-time → (B, C)
        logits = self.fc(x)                   # (B, num_classes)
        return logits

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


In [ ]:
def evaluate(model, data_loader, loss_fn):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    with torch.no_grad():
        for batch_labels, batch_tokens in data_loader:
            batch_labels, batch_tokens = batch_labels.to(device), batch_tokens.to(device)
            outputs = model(batch_tokens)
            loss = loss_fn(outputs, batch_labels)
            preds = outputs.argmax(1)
            total_loss += loss.item() * len(batch_labels)
            total_acc  += (preds == batch_labels).float().sum().item()
            n += len(batch_labels)
    return total_loss / n, total_acc / n

def train_model(model, train_loader, test_loader, epochs=4, lr=1e-3, optimizer_name="adamw"):
    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss":[], "train_acc":[], "test_loss":[], "test_acc":[]}
    start = time.time()

    for epoch in range(1, epochs+1):
        model.train()
        for batch_labels, batch_tokens in train_loader:
            batch_labels, batch_tokens = batch_labels.to(device), batch_tokens.to(device)
            logits = model(batch_tokens)

            loss = loss_fn(logits, batch_labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
        trL, trA = evaluate(model, train_loader, loss_fn)
        teL, teA = evaluate(model, test_loader,  loss_fn)
        history["train_loss"].append(trL); history["train_acc"].append(trA)
        history["test_loss"].append(teL);  history["test_acc"].append(teA)
        print(f"Epoch {epoch}: train {trL:.3f}/{trA:.3f} | test {teL:.3f}/{teA:.3f}")

    sec = round((time.time()-start)/epochs, 3)
    return history, sec


In [ ]:
# --- Run baseline ---
model = TextCNN(vocab_size=vocab_size, embed_dim=100, num_filters=100, num_classes=num_classes).to(device)
print("Params:", f"{count_params(model):,}")
hist_base, sec_base = train_model(model, train_loader, test_loader, epochs=4, lr=1e-3, optimizer_name="adamw")
print(f"Baseline — params={count_params(model):,} | sec/epoch={sec_base} | final acc={hist_base['test_acc'][-1]:.3f}")


Params: 3,030,504
Epoch 1: train 0.319/0.895 | test 0.398/0.859
Epoch 2: train 0.198/0.938 | test 0.337/0.883
Epoch 3: train 0.121/0.967 | test 0.314/0.893
Epoch 4: train 0.078/0.981 | test 0.323/0.896
Baseline — params=3,030,504 | sec/epoch=25.409 | final acc=0.896


In [ ]:
y, x = next(iter(train_loader))
model(x.to(device)).shape

torch.Size([128, 4])

## Grad-CAM for TextCNN

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer_name):
        self.model = model
        self.target_layer_name = target_layer_name
        self.gradients = None
        self.activations = None

        self.hook_layers()

    def hook_layers(self):
        def forward_hook(module, input, output):
            self.activations = output

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0]

        for name, module in self.model.named_modules():
            if name == self.target_layer_name:
                module.register_forward_hook(forward_hook)
                module.register_backward_hook(backward_hook)

    def __call__(self, input_ids, target_category):
        self.model.zero_grad()
        input_ids = input_ids.to(device)
        output = self.model(input_ids.unsqueeze(0))
        output[:, target_category].backward()

        pooled_gradients = torch.mean(self.gradients, dim=[0, 2])
        activations = self.activations[0].detach()

        for i in range(pooled_gradients.size(0)):
            activations[:, i] *= pooled_gradients[i]

        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)
        heatmap /= torch.max(heatmap)

        return heatmap.cpu().numpy()

def visualize_cam(text, model, target_layer_name, target_category, tokenizer, max_len):
    ids = encode_ids(text, max_len=max_len)
    tokens = decode_ids(ids)
    input_ids = torch.tensor(ids, dtype=torch.long)

    grad_cam = GradCAM(model, target_layer_name)
    heatmap = grad_cam(input_ids, target_category)

    if len(heatmap) < len(tokens):
      heatmap = np.pad(heatmap, (0, len(tokens) - len(heatmap)), 'constant')
    elif len(heatmap) > len(tokens):
      heatmap = heatmap[:len(tokens)]

    wrapped_tokens = []
    line_length = 15
    current_line = []
    for i, token in enumerate(tokens):
        current_line.append((token, heatmap[i]))
        if len(current_line) >= line_length or i == len(tokens) - 1:
            wrapped_tokens.append(current_line)
            current_line = []

    plt.figure(figsize=(10, len(wrapped_tokens) * 0.5))
    cmap = plt.cm.Reds
    norm = plt.Normalize(vmin=0, vmax=np.max(heatmap))

    y_offset = len(wrapped_tokens) - 1
    for line in wrapped_tokens:
        for i, (token, intensity) in enumerate(line):
            plt.bar(i, 1, bottom=y_offset, color=cmap(norm(intensity)))
            plt.text(i, y_offset + 0.5, token, ha='center', va='center', color='black' if intensity < 0.5 else 'white', fontsize=9)
        y_offset -= 1

    plt.title(f"Grad-CAM for class {target_category}")
    plt.axis('off')
    plt.show()

text_index = 10
example_text = test_texts[text_index]
example_label = test_labels[text_index]

target_layer = 'conv'

target_category = example_label

print(f"Original text: {example_text}")
print(f"True label: {target_category}")

visualize_cam(example_text, model, target_layer, target_category, wp_tokenizer, MAX_LEN)

In [ ]:
#@title alternative version(made for the MK-CNN model)

def visualize_gradients(model, text):
    model.eval()
    tokens = encode_ids(text)
    input_tensor = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    # Enable gradient tracking on embeddings
    embeds = model.embedding(input_tensor)
    embeds.retain_grad()

    # Forward through rest of network
    x = embeds.transpose(1,2)  # (B,E,T)
    feats = []
    if 3 in model.kernels:
        h3 = torch.max(F.relu(model.conv3(x)), dim=2).values
        feats.append(h3)
    if 4 in model.kernels:
        h4 = torch.max(F.relu(model.conv4(x)), dim=2).values
        feats.append(h4)
    if 5 in model.kernels:
        h5 = torch.max(F.relu(model.conv5(x)), dim=2).values
        feats.append(h5)
    z = torch.cat(feats, dim=1)
    pred_class = z.argmax(dim=1)

    # Backward wrt predicted class
    model.zero_grad()
    z[0, pred_class].backward()

    # Get gradient magnitudes for each word
    grads = embeds.grad.detach().squeeze(0)
    grad_norms = grads.norm(dim=1).to('cpu').numpy()

    # Visualization
    plt.figure(figsize=(20, 2))
    plt.bar(range(len(tokens)), grad_norms, color='salmon')
    plt.xticks(range(len(tokens)), tokens, rotation=45)
    plt.ylabel("Gradient Magnitude")
    plt.title(f"Word importance for: '{text}'")
    plt.show()

# 🔍 Try it
visualize_gradients(mk, "I absolutely love this movie")

### ✍️ Baseline write-up
- Params: `__` | sec/epoch: `__` | final test acc: `__.___`
- Under/overfitting? Point to curves. One concrete next change (what + why).

## Multi-kernel TextCNN (3,4,5)

In [ ]:
class MultiKernelTextCNN(nn.Module):
    """Embedding -> [Conv(k), ReLU, GlobalMax] for k in {3,4,5} -> concat -> Linear"""
    def __init__(self, vocab_size, embed_dim=100, kernels=(3,4,5), num_filters=64, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv3 = nn.Conv1d(embed_dim, num_filters, kernel_size=3, padding=1)
        self.conv4 = nn.Conv1d(embed_dim, num_filters, kernel_size=4, padding=2)
        self.conv5 = nn.Conv1d(embed_dim, num_filters, kernel_size=5, padding=2)
        self.fc    = nn.Linear(num_filters * len(kernels), num_classes)
        self.kernels = kernels

    def forward(self, batch_tokens):
        x = self.embedding(batch_tokens).transpose(1,2)  # (B,E,T)
        feats = []
        if 3 in self.kernels:
            h3 = torch.max(F.relu(self.conv3(x)), dim=2).values
            feats.append(h3)
        if 4 in self.kernels:
            h4 = torch.max(F.relu(self.conv4(x)), dim=2).values
            feats.append(h4)
        if 5 in self.kernels:
            h5 = torch.max(F.relu(self.conv5(x)), dim=2).values
            feats.append(h5)
        z = torch.cat(feats, dim=1)
        return self.fc(z)


In [ ]:
mk = MultiKernelTextCNN(vocab_size=vocab_size, embed_dim=100, kernels=(3,4,5), num_filters=64, num_classes=num_classes).to(device)
print("Params (multi-k):", f"{count_params(mk):,}")
hist_mk, sec_mk = train_model(mk, train_loader, test_loader, epochs=4, lr=1e-3, optimizer_name="adamw")
print(f"Multi-k — params={count_params(mk):,} | sec/epoch={sec_mk} | final acc={hist_mk['test_acc'][-1]:.3f}")


Params (multi-k): 3,077,764
Epoch 1: train 0.276/0.907 | test 0.376/0.868
Epoch 2: train 0.147/0.958 | test 0.314/0.893
Epoch 3: train 0.076/0.982 | test 0.305/0.898
Epoch 4: train 0.038/0.993 | test 0.325/0.902
Multi-k — params=3,077,764 | sec/epoch=25.22 | final acc=0.902


### ✍️ Compare (fill)
| Model    | Kernels  | Params | sec/epoch | Test acc |
|----------|----------|-------:|----------:|---------:|
| Single-k | [3]      |        |           |          |
| Multi-k  | [3,4,5]  |        |           |          |

**Interpret** Did multi-k help? What accuracy/compute trade-off did you observe?


## Text-specific ablations

In [ ]:
# PRESETS = {
#     "k3":      dict(embed_dim=100, kernels=[3],     num_filters=100, opt="adamw", lr=1e-3, max_len=128, pool="max"),
#     #"k5":
#     #"k3_5_7":
#     #"avgpool":
#     #"len64":
#     #"dim200":
# }

# def run_preset(name):
#     import time
#     cfg = PRESETS[name]

#     # 1) Update the global MAX_LEN (encode_ids() uses this) — DataLoader encodes on __getitem__,
#     #    so changing MAX_LEN is enough; no need to recreate loaders.
#     global MAX_LEN
#     MAX_LEN = cfg["max_len"]

#     # 2) Build model according to pooling/kernels
#     if len(cfg["kernels"]) == 1 and cfg["pool"] == "max":
#         m = TextCNN(
#             vocab_size=vocab_size,
#             embed_dim=cfg["embed_dim"],
#             num_filters=cfg["num_filters"],
#             num_classes=num_classes
#         ).to(device)

#     m = MultiKernelTextCNN(
#             vocab_size=vocab_size,
#             embed_dim=cfg["embed_dim"],
#             kernels=tuple(cfg["kernels"]),
#             num_filters=cfg["num_filters"],
#             num_classes=num_classes,
#             pad_idx=PAD_IDX
#         ).to(device)

#     # 3) Train & time it
#     epochs = cfg.get("epochs", 4)
#     t0 = time.time()
#     hist = train_model(
#         m, train_loader, val_loader,
#         epochs=epochs,
#         lr=cfg["lr"],
#         optimizer_name=cfg["opt"],
#         device=device
#     )
#     sec_per_epoch = (time.time() - t0) / max(epochs, 1)

#     # 4) Collect results (use validation accuracy as the comparable metric here)
#     params = sum(p.numel() for p in m.parameters() if p.requires_grad)
#     final_acc = round(hist["val_acc"][-1], 3)

#     return dict(
#         config=name,
#         kernels=cfg["kernels"],
#         pool=cfg["pool"],
#         embed_dim=cfg["embed_dim"],
#         max_len=cfg["max_len"],
#         opt=cfg["opt"],
#         lr=cfg["lr"],
#         params=params,
#         sec_per_epoch=round(sec_per_epoch, 3),
#         final_acc=final_acc
#     )

# # 🔧 Choose which presets to run (students can edit this one line)
# ACTIVE = ["k3", "k5", "avgpool", "k3_5_7", "len64", "dim200"]

# summary = [run_preset(n) for n in ACTIVE]
# summary

### ✍️ Ablation notes (for each ACTIVE)
- Hypothesis → Δ test acc → Mechanism (e.g., “max keeps strongest phrase signal; avg dilutes”).


## Find a method to interpret the text cnn results/cnn filters